In [41]:
%pwd

'd:\\AI Projects\\Rag-Assesment'

In [42]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [43]:
def load_pdf_file(path):
    loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
    return loader.load()

In [44]:
extracted_data = load_pdf_file("D:\AI Projects\Rag-Assesment")

In [45]:
print(len(extracted_data))
print(extracted_data[2232].page_content)

4505
Standard imaging tests, such as x rays, computed
tomography scans (CT scans), and magnetic resonance
imaging (MRI) may be used to check whether the
leukemic cells have invaded other areas of the body,
such as the bones, chest, kidneys, abdomen, or brain.
A gallium scan or bone scan is a test in which a radio-
active chemical is injected into the body. This chemical
accumulates in the areas of cancer or infection, allow-
ing them to be viewed with a special camera.
Treatment
There are two phases of treatment for leukemia.
The first phase is called ‘‘induction therapy.’’ As the
name suggests, during this phase, the main aim of the
treatment is to reduce the number of leukemic cells as
far as possible and induce a remission in the patient.
Once the patient shows no obvious signs of leukemia
(no leukemic cells are detected in blood tests and
bone marrow biopsies), the patient is said to be in
remission. The second phase of treatment is then
initiated. This is called continuation or ma

In [46]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [48]:
text_chunks = text_split(extracted_data)
print("length of text chunks", len(text_chunks))

length of text chunks 40135


In [54]:
from langchain_huggingface import HuggingFaceEmbeddings


In [55]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [56]:
embeddings = download_hugging_face_embeddings()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [58]:
query_result = embeddings.embed_query("hellow")
print("Length :", len(query_result))
print("word : ", query_result)

Length : 384
word :  [-0.10537295788526535, 0.023573847487568855, 0.04428330436348915, 0.031650297343730927, -0.02563256211578846, -0.06927907466888428, 0.05502729490399361, -0.02458062581717968, -0.06484151631593704, -0.016601113602519035, 0.003768637776374817, 0.018087944015860558, 0.006548819597810507, -0.044619567692279816, 0.0395350381731987, 0.019661491736769676, 0.018169140443205833, -0.022650249302387238, -0.17151494324207306, 0.014004288241267204, -0.015144004486501217, 0.025041302666068077, -0.05322638154029846, -0.004285958129912615, -0.057155486196279526, -0.0796857699751854, 0.04419946298003197, 0.08439528942108154, 0.02590228244662285, -0.055372197180986404, -0.006696943659335375, 0.038035593926906586, 0.07572020590305328, -0.006682456936687231, -0.00840506050735712, 0.023866713047027588, -0.09059397131204605, -0.11229240894317627, 0.0348728708922863, 0.028340838849544525, 0.007666461635380983, -0.061546117067337036, 0.022727102041244507, -0.020095637068152428, 0.03197801

In [71]:
from dotenv import load_dotenv
import os
load_dotenv()

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")


In [63]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name="test"

# if index_name not in pc.list_indexes():
#     pc.create_index(
#         name=index_name,
#         dimension=384,
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1")
#     )

In [ ]:
# Uploads documentions to vector db do only once

from langchain_pinecone import PineconeVectorStore

doSearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [67]:
doSearch

In [68]:
retriever = doSearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [69]:
retrieved_doc = retriever.invoke("what is acne ?")

In [70]:
retrieved_doc

[Document(id='8f598cfc-2cc0-45c9-b382-04baeac3992d', metadata={'author': '', 'creationDate': "D:20061016201933+02'00'", 'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'file_path': 'D:\\AI Projects\\Rag-Assesment\\book.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': "D:20061016220345+02'00'", 'moddate': '2006-10-16T22:03:45+02:00', 'page': 269.0, 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'D:\\AI Projects\\Rag-Assesment\\book.pdf', 'subject': '', 'title': '', 'total_pages': 4505.0, 'trapped': ''}, page_content='forms of acne.\nPurpose\nDifferent types of antiacne drugs are used for\ndifferent purposes. For example, lotions, soaps, gels,\nand creams containing benzoyl peroxide or tretinoin\nmay be used to clear up mild to moderately severe\nacne. Isotretinoin (Accutane) is prescribed only for\nvery severe, disfiguring acne.\nAcne is a skin condition that occurs when pores or\nhair follicles become blocked. This blockage allows a\nwaxy material c

In [72]:
import cohere
from langchain_cohere import ChatCohere

# Raw Cohere client (optional if you also need SDK)
co = cohere.ClientV2(COHERE_API_KEY)

# Wrap Cohere as a LangChain LLM
llm = ChatCohere(model="command-r", temperature=0)

In [ ]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


system_prompt = (
    "You are a medical assistant. "
    "Answer ONLY using the provided context. "
    "If the answer is not in the context, say: "
    "'I'm sorry, I don't have information about that.' "
    "Never make up answers.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [78]:
# 3) Helper to convert retrieved docs -> one string
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs,
        "input": itemgetter("input"),
    }
    | prompt
    | llm
)

NameError: name 'prompt' is not defined